# Local RAG with Small Language Models

A Retrieval-Augmented Generation (RAG) pipeline built on small language models,
running entirely on a Raspberry Pi 5 with no cloud dependency.

## What this notebook does

Instead of relying on a model's memory alone, RAG retrieves relevant text from
a document collection and gives it to the model as context. This notebook walks
through that flow step by step:

1. Load the source PDF and split it into pages, sentences and chunks
2. Convert each chunk into an embedding (sentence-transformers)
3. Save the embeddings to disk and load them into a vector database (LanceDB)
4. Index the vectors and search them with a question
5. Build a prompt that combines the question with the retrieved chunks
6. Generate an answer with a small language model served by Ollama

## Tech stack

<table style="margin-left: 0; margin-right: auto;">
  <thead>
    <tr>
      <th style="text-align: left;">Component</th>
      <th style="text-align: left;">Role</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td style="text-align: left;">PyMuPDF</td>
      <td style="text-align: left;">Extracts the text from the PDF</td>
    </tr>
    <tr>
      <td style="text-align: left;">spaCy</td>
      <td style="text-align: left;">Splits the text into sentences</td>
    </tr>
    <tr>
      <td style="text-align: left;">sentence-transformers</td>
      <td style="text-align: left;">Creates the text embeddings</td>
    </tr>
    <tr>
      <td style="text-align: left;">LanceDB</td>
      <td style="text-align: left;">Vector database for retrieval</td>
    </tr>
    <tr>
      <td style="text-align: left;">Ollama (phi3-mini)</td>
      <td style="text-align: left;">Runs the small language model locally</td>
    </tr>
    <tr>
      <td style="text-align: left;">Raspberry Pi 5</td>
      <td style="text-align: left;">Target hardware</td>
    </tr>
  </tbody>
</table>

## Credits

Adapted from Chapter 13 of *Domain-Specific Small Language Models*. The chapter's
example builds on the research paper *DeepSpeed-FastGen: High-throughput Text
Generation for LLMs via MII and DeepSpeed-Inference*. This version replaces
llama-cpp-python with Ollama and the in-memory dataframe search with LanceDB so
it runs on modest hardware.

In [17]:
# Standard library
import re

# Third-party
import ollama
import lancedb
import numpy as np
import pyarrow as pa
from tqdm import tqdm
from spacy.lang.en import English
from sentence_transformers import SentenceTransformer

# Local helpers
from rag_helpers import (
    open_and_read_pdf,
    split_list,
    load_or_build_embeddings_df,
    prompt_formatter,
)

## Step 1: Load the source document

Every RAG pipeline starts with a knowledge source. Here it is the research
paper *DeepSpeed-FastGen: High-throughput Text Generation for LLMs via MII and
DeepSpeed-Inference*.

The next cell does two things:

1. **Points to the PDF** stored in the `data/` folder.
2. **Reads it page by page** with `open_and_read_pdf()`, a helper from
   `rag_helpers.py`. It extracts and cleans the text of each page and
   calculates basic statistics along the way.

In [2]:
pdf_path = "data/deepspeed_fastgen_paper.pdf"
pages_and_texts = open_and_read_pdf(pdf_path)

100%|██████████████████████████████████████████████████| 14/14 [00:00<00:00, 83.71it/s]


`pages_and_texts` holds one dictionary per page with the cleaned text and
simple statistics (characters, words, sentences and estimated tokens). The
token estimate divides the character count by 4, a rule of thumb for English
text, not an exact tokenizer count. We will use these numbers later to decide
how to split the text into chunks.

## Step 2: Split each page into sentences

The page statistics from Step 1 used a crude sentence count (splitting on
`". "`). Here we do it properly with spaCy.

Sentences matter because the next steps group them into chunks small enough
to embed and retrieve. Splitting cleanly first means we never cut a chunk in
the middle of a sentence.

We use spaCy's `sentencizer`, a lightweight rule-based component that detects
sentence boundaries from punctuation. It needs no model download and runs
fast, which suits a Raspberry Pi.

In [3]:
# Create a blank English pipeline (tokenizer only, no trained model).
nlp = English()

# Add the rule-based sentence splitter to the pipeline.
nlp.add_pipe("sentencizer")

for item in tqdm(pages_and_texts):
    # Run the page text through the pipeline and collect its sentences.
    item["sentences"] = list(nlp(item["text"]).sents)

    # spaCy returns Span objects; convert them to plain strings so they are
    # easy to store, inspect and embed later.
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]

    # Record the sentence count so we can compare it with the raw estimate.
    item["page_sentence_count_spacy"] = len(item["sentences"])

100%|█████████████████████████████████████████████████| 14/14 [00:00<00:00, 233.97it/s]


Each page dictionary now has two new entries:

- `sentences`: the page text as a list of sentence strings.
- `page_sentence_count_spacy`: the number of sentences spaCy found.

This count is more reliable than the earlier `page_sentence_count_raw`, which
only split on `". "` and would miscount abbreviations or numbered lists.


## Step 3: Group sentences into chunks

A whole page is often too long to embed or to hand to the model in one piece.
Two limits apply:

- The **embedding model** has a maximum number of tokens it can process.
- The **generator model** has a context window, which has to hold the
  question, the retrieved text and the answer.

To stay within both, we split each page's sentences into small groups, or
chunks. The helper `split_list()` does the splitting: it cuts a list into
sublists of a target size, and the last sublist may be shorter if the items
don't divide evenly.

In [4]:
# Maximum number of sentences per chunk. This is a tunable setting: smaller
# chunks are more precise to retrieve, larger ones carry more context.
num_sentence_chunk_size = 10

for item in tqdm(pages_and_texts):
    # Split each page's sentences into groups of up to 10 using the helper.
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=num_sentence_chunk_size)

    # Record how many chunks the page produced.
    item["num_chunks"] = len(item["sentence_chunks"])

100%|██████████████████████████████████████████████| 14/14 [00:00<00:00, 117912.16it/s]


Each page dictionary gained two entries:

- `sentence_chunks`: the page's sentences grouped into lists of up to 10.
- `num_chunks`: how many groups the page produced.

The chunk size is stored in `num_sentence_chunk_size`, so it is easy to change
and rerun.

## Step 4: Build one record per chunk

So far the data is organized by page. For retrieval, each chunk needs to be an
independent record: the chunk text plus metadata, such as the page it came from
and its size. That is what gets embedded and searched.

The next cell flattens the pages into a new list, `pages_and_chunks`, with one
dictionary per chunk. Along the way it joins each chunk's sentences back into a
single string and tidies the spacing.

In [5]:
pages_and_chunks = []

for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}

        # Keep track of the source page so answers can be traced back to it.
        chunk_dict["page_number"] = item["page_number"]

        # Join the chunk's sentences into one string, replace non-breaking
        # spaces (common in PDFs) with regular spaces and trim the ends.
        joined_sentence_chunk = "".join(sentence_chunk).replace("\xa0", " ").strip()

        # Add a missing space after a period that is directly followed by a
        # capital letter (".Next" becomes ". Next").
        joined_sentence_chunk = re.sub(r'\.([A-Z])', r'. \1', joined_sentence_chunk)

        chunk_dict["sentence_chunks"] = joined_sentence_chunk

        # Chunk statistics, using the same approximations as before.
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4  # ~4 chars per token

        pages_and_chunks.append(chunk_dict)

100%|███████████████████████████████████████████████| 14/14 [00:00<00:00, 11502.50it/s]


`pages_and_chunks` is a flat list with one dictionary per chunk, holding:

- `page_number`: the source page.
- `sentence_chunks`: the cleaned chunk text, as a single string.
- `chunk_char_count`, `chunk_word_count`, `chunk_token_count`: size statistics.

The token count is again an estimate (characters divided by 4). It is useful
for checking that chunks fit within the embedding model's token limit. This
list is what we will embed in the next step.

Note: `sentence_chunks` here holds the chunk as a single joined string, unlike
the `sentence_chunks` key on the page dictionaries from Step 3, which holds a
list of sentence groups.

## Step 5: Load the embedding model

Computers can't compare the meaning of two text chunks directly, so we convert
each chunk into an **embedding**: a list of numbers (a vector) that captures
what the text means. Chunks with similar meaning end up close together in that
vector space, which is what makes search by meaning possible, instead of
search by exact keywords. Closeness is measured with a metric such as cosine
similarity.

We use the [sentence-transformers](https://github.com/UKPLab/sentence-transformers/)
library, an open source Python package that computes these vectors for
sentences and paragraphs. Its models are BERT-family encoders hosted on the
Hugging Face Hub.

The model we load is
[`all-mpnet-base-v2`](https://huggingface.co/sentence-transformers/all-mpnet-base-v2).
It maps text to a 768-dimensional vector and was fine-tuned on about a billion
sentence pairs, which makes it a good fit for semantic search and clustering.
It is small enough to run on a CPU, so no GPU is needed.

> **Note: contrastive learning.** Embedding models like this one are trained by
> showing them pairs of texts. Pairs that belong together (positive pairs) are
> pulled closer in vector space, and pairs that don't (negative pairs) are
> pushed apart. This is an increasingly popular way to specialize small
> embedding models for RAG systems.

In [6]:
# Load the embedding model. If it isn't on disk yet, the checkpoint is
# downloaded from the Hugging Face Hub the first time and cached for later runs.

embedding_model = SentenceTransformer(
    model_name_or_path="all-mpnet-base-v2",
    device="cpu")  # run on the CPU; (the Raspberry Pi has no GPU)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

`embedding_model` is now ready to turn text into 768-dimensional vectors. We
haven't used it yet. In the next step we will pass our chunks through it.

The first run needs an internet connection to download the model. After that
it is cached locally, so the notebook can run offline.

## Step 6: Generate the embeddings

Now we apply the embedding model to every chunk. Each chunk's text is converted
into a 768-dimensional vector and stored in the same dictionary under a new key,
`embedding`, so the text and its vector stay together.

This is the most time-consuming step so far, because the model has to process
every chunk. On a Raspberry Pi 5 it runs on the CPU, so expect it to take a
while for larger documents. The progress bar shows how far along it is.

In [7]:
# Collect the text of every chunk into one list.
chunk_texts = [item["sentence_chunks"] for item in pages_and_chunks]

# Encode all chunks in batches. This is usually faster than encoding one chunk
# at a time, and the progress bar is built in.
embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True)

# Store each vector next to its chunk so text and embedding stay together.
for item, embedding in zip(pages_and_chunks, embeddings):
    item["embedding"] = embedding

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Every dictionary in `pages_and_chunks` now has an `embedding` key holding the
vector for its chunk. These vectors are what we will store in a vector
database and search to find the chunks most relevant to a question.

## Step 7: Save the embeddings to disk

Generating embeddings is the slowest part of the pipeline, and the result
doesn't change unless the document or the model does. So we save it once and
reuse it.

The helper `load_or_build_embeddings_df()` (in `rag_helpers.py`) works like a
cache:

- **If the CSV file exists**, it loads it and converts each embedding from text
  back into a NumPy array.
- **If it doesn't**, it builds a DataFrame from `pages_and_chunks`, saves it as
  a CSV and returns it.

The book keeps the embeddings in memory and searches them directly, to show how
embedding search works under the hood. In this version the CSV only serves as a
checkpoint: the searchable index lives in a vector database, which comes next.

In [8]:
# On the first run this saves the CSV; on later runs it loads it, so the
# embedding cell above doesn't need to be rerun after a restart.
# globals().get() returns None instead of an error when `pages_and_chunks`
# doesn't exist, which is the case after a restart that skips the earlier cells.

text_chunks_and_embeddings_df = load_or_build_embeddings_df(
    "text_chunks_and_embeddings_df.csv",
    pages_and_chunks=globals().get("pages_and_chunks"),
)

`text_chunks_and_embeddings_df` is a table with one row per chunk: the page
number, the chunk text, its size statistics and its embedding vector. The CSV
file now sits next to the notebook, so the embeddings survive a kernel restart.

## Step 8: Create the vector database

Until now the embeddings live in a table in memory (and in a CSV checkpoint).
That is fine for one small paper, but searching by comparing a question against
every row doesn't scale. A **vector database** stores the embeddings in an
indexed form built for similarity search, and it keeps them on disk.

We use [LanceDB](https://lancedb.github.io/lancedb/), an open source vector
database that is **embedded**: it runs inside our Python process and stores its
data in a local folder. There is no server to install or manage, which suits a
Raspberry Pi and keeps all data on the machine.

The next cell opens a connection to a database folder named `paperdb`. If the
folder doesn't exist yet, LanceDB creates it.

In [9]:
# Connect to a local LanceDB database stored in the folder "paperdb".
# The folder is created automatically on the first run.

db = lancedb.connect("paperdb")

`db` is a connection to the database. It is empty for now. In the next step we
create a table inside it and load our chunks and embeddings.

## Step 9: Prepare the embeddings for the database

Before loading the chunks into LanceDB, each embedding needs to be in a clean
numeric format. This cell converts every vector in the `embedding` column to a
32-bit floating-point NumPy array, the format vector databases normally expect,
and stores the result in a new column, `embedding_final`.

The next step uses `embedding_final` as the vector column of the database table.

In [10]:
# Convert each embedding to a float32 NumPy array, the format vector databases
# normally expect.

text_chunks_and_embeddings_df["embedding_final"] = text_chunks_and_embeddings_df["embedding"].apply(
    lambda x: np.asarray(x, dtype=np.float32))

The table has one new column, `embedding_final`, holding the same vectors as
32-bit NumPy arrays, ready for the database. The original `embedding` column is
unchanged.

## Step 10: Reshape the data for the database

LanceDB can create a table from a dictionary where each key is a column name
and each value is the list of values for that column. Our data is currently
organized as a table with one row per chunk, so this cell reshapes it in two
stages:

1. **Select the columns we need.** For each chunk we keep the page number, the
   size statistics, the chunk text and the final embedding. The temporary
   `embedding_str` column and the original `embedding` column are left behind.
2. **Pivot from rows to columns.** The list of per-chunk records becomes a
   dictionary with one list per column.

The result, `data_dict`, is what we load into the database in the next step.

In [11]:
# Stage 1: build one small dictionary per chunk, keeping only the fields
# we want to store in the database.
final_data = []
for row in text_chunks_and_embeddings_df.itertuples(index=False):
    temp = {}
    temp["page_number"] = row.page_number
    temp["chunk_char_count"] = row.chunk_char_count
    temp["chunk_word_count"] = row.chunk_word_count
    temp["chunk_token_count"] = row.chunk_token_count
    temp["sentence_chunks"] = row.sentence_chunks
    temp["embedding"] = np.array(row.embedding_final)  # the cleaned vector from Step 9
    final_data.append(temp)

# Stage 2: pivot from a list of records to one list per column.
# Every record has the same keys, so we take them from the first one.
data_dict = {}
for key in final_data[0].keys():
    data_dict[key] = [d[key] for d in final_data]

`data_dict` has one key per column (`page_number`, `chunk_char_count`,
`chunk_word_count`, `chunk_token_count`, `sentence_chunks` and `embedding`).
Each key holds a list with one value per chunk, in the same order. Note that
the vector column is now called `embedding` again, but it contains the cleaned
vectors from the previous step.

## Step 11: Create the table and load the data

Now we create a table inside the database and fill it with our chunks and
embeddings. We pass two things:

- **The data**: `data_dict` from the previous step, converted into a PyArrow
  table, the columnar format LanceDB uses internally.
- **A schema**: an explicit definition of each column's type. Declaring it
  ourselves means the table has exactly the types we expect, instead of
  letting LanceDB guess them.

The most important column is `embedding`. It is declared as a fixed-size list
of 768 32-bit floats, matching the output size of the `all-mpnet-base-v2`
model. LanceDB needs this fixed size to index the vectors for fast search.

With `mode="overwrite"`, running the cell again replaces the table instead of
failing because it already exists. That keeps the notebook easy to rerun.

In [12]:
table = db.create_table(
    "paper_embeddings_table",              # name of the table inside the database
    data=pa.Table.from_pydict(data_dict),  # our columns, converted to a PyArrow table
    mode="overwrite",                      # replace the table if it already exists
    schema=pa.schema([
        ("page_number", pa.int64()),
        ("chunk_char_count", pa.int64()),
        ("chunk_word_count", pa.int64()),
        ("chunk_token_count", pa.float64()),   # float because it is characters / 4
        ("sentence_chunks", pa.string()),      # the chunk text
         # Fixed-size vector column. The 768 is tied to the embedding model:
        # it is the output size of all-mpnet-base-v2. If the model changes,
        # this number must change with it.
        ("embedding", pa.list_(pa.float32(), list_size=768)),
    ]),
)

[2026-11-02T22:02:03Z WARN  lance::dataset::write::insert] No existing dataset at /home/ezzio/Documents/repo/local-slm-rag/paperdb/paper_embeddings_table.lance, it will be created


`table` is a handle to the new table, `paper_embeddings_table`, which now holds
one row per chunk. The data is stored on disk in the `paperdb` folder, so it is
still there after a restart and can be opened again without recomputing
anything. In the next step we search it.

## Step 12: Index the embeddings

The table now holds our vectors, but searching it still means comparing a
question against every row. An **index** organizes the vectors so the database
can find the closest ones without checking them all. This matters little for
one small paper, but it is what makes vector search scale to thousands or
millions of chunks.

The cell below sets three options:

- **`metric="cosine"`**: how closeness is measured. Cosine similarity compares
  the direction of two vectors and ignores their length, the usual choice for
  text embeddings.
- **`vector_column_name="embedding"`**: which column holds the vectors.
- **`index_type="IVF_FLAT"`**: the index type. IVF (inverted file) groups
  similar vectors into clusters, and a search only looks inside the clusters
  closest to the question. FLAT means the vectors inside each cluster are
  stored as they are, without compression, so results stay accurate.

In [13]:
# Build a search index on the embedding column of our table.
db["paper_embeddings_table"].create_index(
    metric="cosine",                # measure closeness with cosine similarity
    vector_column_name="embedding", # the column that holds the vectors
    index_type="IVF_FLAT",          # cluster the vectors; keep them uncompressed
)

/tmp/ipykernel_106920/3561296246.py:2: DeprecationWarning: The create_index() API with metric/num_partitions parameters is deprecated and will be removed in a future version. Please migrate to the new unified API:
  # Old (deprecated):
  table.create_index('l2', vector_column_name='my_vector')
  # New (recommended):
  table.create_index('my_vector', config=IvfPq(distance_type='l2'))
  db["paper_embeddings_table"].create_index(


The table `paper_embeddings_table` now has a vector index on its `embedding`
column. Nothing is returned, because the index is stored with the table in the
`paperdb` folder. Searches in the next step will use it automatically.

If rows are added to the table later, the index should be rebuilt so the new
rows are covered.

## Step 13: Search the database

Now the pipeline can answer its first question. Searching has two parts:

1. **Embed the question.** The question is converted into a vector with the
   same embedding model used for the chunks. Using the same model matters,
   because vectors from different models can't be compared.
2. **Find the nearest chunks.** The database returns the chunks whose vectors
   are closest to the question's vector, measured with cosine similarity, the
   same metric the index was built with.

The example query, "blocked KV-cache", is a topic mentioned in the original
paper. We ask for the 3 closest chunks and keep them in `context_items`, the
context that will later be given to the language model.

In [14]:
# The question we want to answer from the paper.
query = "blocked KV-cache"

# Embed the question with the same model used for the chunks.
# convert_to_tensor=False returns a NumPy array, which LanceDB accepts.
query_embedding = embedding_model.encode(query, convert_to_tensor=False)

results = (
    db["paper_embeddings_table"]
    .search(query_embedding, vector_column_name="embedding")
    .metric("cosine")  # same metric as the index built in Step 12
    .limit(3)          # keep the 3 closest chunks
    .to_list()         # return the matches as a list of dictionaries
)

# These retrieved chunks become the context for the language model.
context_items = results

`context_items` is a list of 3 dictionaries, one per matching chunk, ordered
from closest to farthest. Each holds the chunk text (`sentence_chunks`), its
page number, its size statistics and a `_distance` score. A lower distance means
the chunk is closer in meaning to the question.

These chunks are the "retrieval" in RAG. In the next steps we place them in a
prompt next to the question, so the model answers from the paper instead of
from memory.

## Step 14: Build the prompt and generate the answer

The language model can't see our database, so we hand it the retrieved chunks
ourselves. The helper `prompt_formatter()` (in `rag_helpers.py`) combines three
things into one prompt:

- **Instructions** telling the model to answer from the context.
- **The retrieved chunks**, listed as bullet points.
- **The user's question**, followed by `Answer:` so the model knows where to
  begin.

This is the "augmented" part of Retrieval-Augmented Generation.

The prompt is then sent to a small language model served locally by
[Ollama](https://ollama.com/), the "generation" part. The book's example uses
`llama-cpp-python` for this step. Ollama does the same job with a simpler
setup, which suits the Raspberry Pi.

Two options control the model's behavior:

- **`num_ctx`**: the context window, the maximum number of tokens the model
  can handle at once (prompt plus answer). 4096 is enough for the question and
  three chunks, and keeps memory use modest.
- **`stop`**: text sequences that make the model stop writing as soon as it
  produces them. Here it stops at the first line break, which keeps the answer
  short and speeds up generation on the Pi.

Ollama must be running, and the model must already be downloaded
(`ollama pull phi3-mini`).

In [20]:
# The small language model that will write the answer. It must already be
# pulled in Ollama.
MODEL = "phi3-mini:latest"

# Combine the question and the retrieved chunks into one prompt.
prompt = prompt_formatter(query=query, context_items=context_items)

# Send the prompt to the model running locally in Ollama.
r = ollama.chat(
    model=MODEL,
    messages=[{"role": "user", "content": prompt}],
    options={
        "num_ctx": 4096,         # context window: prompt and answer must fit in it
        "stop": ["Q:", "\n"],    # stop at a new question or at the first line break
    },
)

# The generated text is inside the response's "message" field.
answer = r["message"]["content"]
print("Response:\n\n", answer)

Response:

  A blocked KV-cache, as discussed in the context, increases system throughput by increasing the amount of potential sequence concurrency. It does so by eliminating KV-cache induced memory fragmentation. This means that instead of allowing the KV-cache to create non-contiguous memory spaces for storing data (which can slow down the system due to increased memory fragmentation), a blocked KV-cache is designed in such a way that it creates a more contiguous memory layout. This contiguous layout allows for higher concurrency because it enables the system to process multiple sequences or requests more efficiently, without the overhead caused by fragmented memory spaces.
